In [1]:
import pandas as pd
import numpy as np

In [3]:
df = pd.read_csv("SMSSpamCollection", sep="\t", names=["label", "message"])
df["label_num"] = df["label"].map({"ham": 0, "spam": 1})
df["MessageId"] = range(len(df))

df.head()

,label,message,label_num,MessageId
0,ham,"Go until jurong point, crazy.. Available only ...",0,0
1,ham,Ok lar... Joking wif u oni...,0,1
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,2
3,ham,U dun say so early hor... U c already then say...,0,3
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,4


In [4]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5572 entries, 0 to 5571
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   label      5572 non-null   object
 1   message    5572 non-null   object
 2   label_num  5572 non-null   int64 
 3   MessageId  5572 non-null   int64 
dtypes: int64(2), object(2)
memory usage: 174.3+ KB


In [6]:
df.shape

(5572, 4)

In [7]:
df.describe()

,label_num,MessageId
count,5572.000000,5572.000000
mean,0.134063,2785.500000
std,0.340751,1608.642181
min,0.000000,0.000000
25%,0.000000,1392.750000
50%,0.000000,2785.500000
75%,0.000000,4178.250000
max,1.000000,5571.000000


In [8]:
df.head()

,label,message,label_num,MessageId
0,ham,"Go until jurong point, crazy.. Available only ...",0,0
1,ham,Ok lar... Joking wif u oni...,0,1
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1,2
3,ham,U dun say so early hor... U c already then say...,0,3
4,ham,"Nah I don't think he goes to usf, he lives aro...",0,4


In [9]:
df.isnull().sum()

,0
label,0
message,0
label_num,0
MessageId,0


In [10]:
from sklearn.model_selection import StratifiedKFold, train_test_split

X = df["message"]
y = df["label_num"]

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,stratify=y,random_state=0)


# Train test split

In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words="english", max_features=3000)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


# Model comparison naive bayes, logistic regression, svc

In [12]:
#naive baiyes, logistic regression, svc
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

nb = MultinomialNB()
lr = LogisticRegression(max_iter=1000)
svm = LinearSVC()

# AdaBoost with Decision Stumps

In [13]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import VotingClassifier, StackingClassifier, AdaBoostClassifier


stump = DecisionTreeClassifier(max_depth=1)

adaboost = AdaBoostClassifier(estimator=stump,n_estimators=50,learning_rate=1.0,random_state=0)

In [14]:
#voting classifier
voting_hard = VotingClassifier(
    estimators=[
        ("nb", nb),
        ("lr", lr),
        ("svm", svm)
    ],
    voting="hard"
)

voting_soft = VotingClassifier(
    estimators=[
        ("nb", nb),
        ("lr", lr),
        ("svm", LogisticRegression(max_iter=1000))
    ],
    voting="soft"
)

In [15]:
#stacking classifier

stacking = StackingClassifier(
    estimators=[
        ("nb", nb),
        ("lr", lr),
        ("svm", LogisticRegression(max_iter=1000))
    ],
    final_estimator=LogisticRegression()
)

# stratified k-fold

In [16]:
from sklearn.metrics import (confusion_matrix,precision_score,recall_score,f1_score,roc_auc_score,accuracy_score)

In [17]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_model(model, X, y):

    precision_list = []
    recall_list = []
    f1_list = []
    roc_list = []

    for train_index, test_index in skf.split(X, y):

        X_train_fold = X[train_index]
        X_test_fold = X[test_index]

        y_train_fold = y.iloc[train_index]
        y_test_fold = y.iloc[test_index]

        model.fit(X_train_fold, y_train_fold)

        y_pred = model.predict(X_test_fold)

        precision_list.append(precision_score(y_test_fold, y_pred))
        recall_list.append(recall_score(y_test_fold, y_pred))
        f1_list.append(f1_score(y_test_fold, y_pred))

        try:
            y_prob = model.predict_proba(X_test_fold)[:,1]
            roc_list.append(roc_auc_score(y_test_fold, y_prob))
        except:
            roc_list.append(0)

    return {
        "precision_mean": np.mean(precision_list),
        "precision_std": np.std(precision_list),
        "recall_mean": np.mean(recall_list),
        "recall_std": np.std(recall_list),
        "f1_mean": np.mean(f1_list),
        "f1_std": np.std(f1_list),
        "roc_mean": np.mean(roc_list),
        "roc_std": np.std(roc_list)
    }

In [18]:
models = {
    "Naive Bayes": nb,
    "Logistic Regression": lr,
    "SVM": svm,
    "Voting Hard": voting_hard,
    "Voting Soft": voting_soft,
    "Stacking": stacking,
    "AdaBoost (Stumps)": adaboost
}

results = []

for name, model in models.items():

    print("evaluating:", name)

    res = evaluate_model(model, X_train_tfidf, y_train)

    res["model"] = name

    results.append(res)

results_df = pd.DataFrame(results)

results_df

evaluating: Naive Bayes
evaluating: Logistic Regression
evaluating: SVM
evaluating: Voting Hard
evaluating: Voting Soft
evaluating: Stacking
evaluating: AdaBoost (Stumps)


,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,roc_mean,roc_std,model
0,0.998000,0.004000,0.851162,0.022664,0.918615,0.014128,0.991512,0.002857,Naive Bayes
1,0.987257,0.014423,0.667227,0.037260,0.795866,0.029891,0.991480,0.002597,Logistic Regression
2,0.983158,0.018428,0.877913,0.032546,0.927306,0.023028,0.000000,0.000000,SVM
3,0.990154,0.010956,0.841120,0.020606,0.909458,0.014548,0.000000,0.000000,Voting Hard
4,0.993719,0.005138,0.790952,0.026588,0.880569,0.016710,0.993114,0.002422,Voting Soft
5,0.976119,0.012854,0.894608,0.026400,0.933494,0.019469,0.993153,0.002466,Stacking
6,0.936640,0.028858,0.309552,0.050508,0.463553,0.060118,0.901853,0.010393,AdaBoost (Stumps)


Based on these results, the Stacking classifier is the best combining strategy for this problem. It achieved the highest F1-score (0.933), highest recall (0.895), and strong ROC-AUC (0.993), indicating excellent overall performance

In [20]:
results_df.to_csv("ensemble_comparison.csv", index=False)

print("saved ensemble_comparison.csv")

saved ensemble_comparison.csv


In [21]:
final_model = stacking

final_model.fit(X_train_tfidf, y_train)

y_pred = final_model.predict(X_test_tfidf)

try:
    y_prob = final_model.predict_proba(X_test_tfidf)[:,1]
except:
    y_prob = np.zeros(len(y_pred))

In [22]:
cm = confusion_matrix(y_test, y_pred)

precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

print("\nPrecision:", precision)
print("Recall:", recall)
print("F1 Score:", f1)

Confusion Matrix:
[[959   7]
 [ 17 132]]

Precision: 0.9496402877697842
Recall: 0.8859060402684564
F1 Score: 0.9166666666666666


In [23]:
output_df = pd.DataFrame({
    "MessageId": y_test.index,
    "Actual": y_test.values,
    "Predicted": y_pred,
    "Probability": y_prob
})

output_df.to_csv("final_predictions.csv", index=False)


Stacking classifier performed best overall because it learns how to optimally combine multiple base models using a meta-learner. Unlike voting, which treats models equally or averages probabilities, stacking learns which model is more reliable in different situations. AdaBoost also performed well because it focuses on correcting mistakes using decision stumps, but stacking was more stable and achieved better overall performance. Therefore, stacking is the recommended combining strategy because it provides higher accuracy, better generalization, and improved robustness compared to voting and boosting methods.